In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'medium',
        'figure.figsize': (18, 6),
        'axes.labelsize': 'medium',
        'axes.titlesize':'medium',
        'xtick.labelsize':'medium',
        'ytick.labelsize':'medium'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import time
import datetime
import pytz

NYC_tz = pytz.timezone("America/New_York") 

import sys
sys.path.append("../../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP, _alias_to_cusip
from TB.FixedRateBondsTB import FixedRateBondsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedStructure, UnifiedValue
from TB.TimeseriesBuilder import TimeseriesBuilder

curve_mdp = IRSwapsMDP(source="GSQUANT-RL")
usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-RL")

ts_builder = TimeseriesBuilder()

In [3]:
start = datetime.date(2010, 1, 1)
end = datetime.date(2026, 4, 7)

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[
  		UnifiedQuery(
            curve="USD-OIS",
            tenor="2y2y/5y5y",
            value=UnifiedValue.IRS_RATE,
        ),
        # UnifiedQuery(
        #     cusip="CT10",
        #     value=UnifiedValue.FRB_YTM,
        # ),       
    ],
    n_jobs=12,
	routers={
        "IRS": IRSwapsTB(curve_mdp, show_tqdm=True),
		# "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
	},
    ignore_cache_miss=True,
)
df

WARNING	Task(Task-2) Caching.computed_timeseries_store:computed_timeseries_store.py:_open_duckdb_graceful()- DuckDB unavailable (file locked), falling back to parquet-only: C:\Users\chris\clee\ARBS\data\ts\computed_ts.duckdb
LOADING USD-OIS raw curves...:   0%|          | 0/4243 [00:00<?, ?it/s]         

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,USD-OIS 2y2y/5y5y CURVE RATE
Date,
2010-01-04,153.298395
2010-01-05,154.059612
2010-01-06,161.596166
2010-01-07,158.280302
2010-01-08,159.322012
...,...
2026-03-23,59.768605
2026-03-24,54.636458
2026-03-25,56.631277


In [4]:
plot, fig, ax, ax2, legend = make_secondary_axis_plot(engine="plotly")
plot(
    df["USD-OIS 2y2y/5y5y CURVE RATE"],
    which="left",
    indicators=[
        # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
        # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
    ],
    # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
)
# plot(
#     df["CT10 OUTRIGHT YTM"],
#     which="left",
#     indicators=[
#         # {"kind": "last", "style": {"linestyle": "--", "linewidth": 1.2, "color": "tab:purple"}},  
#         # {"kind": "sma", "window": 60, "style": {"linestyle": "--", "color": "green"}},
#     ],
#     # ou={"enable": True, "steps": 126, "add_metrics_to_legend": True}
# )
legend(show_date=True, loc="upper left")
plt.show()